# Exploring Uncertainty in Climate Model Precipitation

There are several kinds of scientific uncertainty that arise when working with long-term projections of future climates:

1. Model Uncertainty, which illustrates the differences between different models (namely, how model physics, settings, or parameters can change the outcome)
2. Scenario Uncertainty, which illustrates the differences in outcomes between end-of-century emissions targets
3. Internal Variability, which represents the variations inherent within the climate system itself

This notebook is divided into two analysis components which explore the following, respectively:
- <b>Model Uncertainty</b> in the Cal-Adapt: Analytics Engine by focusing on <b>extreme precipitation trends</b> across simulations. We also compare the suite of models currently available in the Cal-Adapt Data Catalog to the full set of CMIP6 models to illustrate the differences between our models and all available models.
- <b>Internal Variability</b> in the Cal-Adapt: Analytics Engine by focusing on <b>extreme precipitation trends</b> across <i>ensemble members</i>\*. We also show where the Cal-Adapt: Analytics Engine <i>downscaled</i>\** ensemble members fit within the larger ensemble.

\* Ensemble member: When a given model is run multiple times, we call the group of runs an <i>ensemble</i>. Each ensemble member represents a distinct realization featuring its own combination of model parameters. We call the the differences between ensemble members <b>Internal Variability</b>. 

\** Downscaling: One ensemble member from each Cal-Adapt: Analytics Engine model was used to provide input data for the Weather Research and Forecasting (WRF) Model, allowing us to capture much smaller-scale details than the CMIP6 models. 

(want to insert the table from my storyboard here)

For this demo, we choose to focus on <b>Sonoma County</b>, which is vulnerable to flooding along the Russian River.

---

<b>Story</b>

As a user, I want to understand when <span style="color:red"><b>it is appropriate to use only the downscaled Analytics Engine models to draw conclusions about extreme precipitation statistics</b></span>, by visualizing the differences across models and within model ensembles. In this notebook we will show you:

1. A range of possibilities for the end of century across all available CMIP6 models
2. Which models are provided in the Analytics Engine as compared to the rest of CMIP6 models
3. The results for each model's set of ensemble members
4. Where the downscaled results for a given model fit within the larger ensemble

Part 0 -- Setup the notebook, including CMIP6 processing code

Part 1 -- Model Uncertainty:
- Part 1.1 -- Get and process the CMIP6 data; compute statistics
- Part 1.2 -- Visualize annual maximum precipitation through end-of-century across all CMIP6 models
- Part 1.3 -- Highlight the Analytics Engine models within this spread
- Part 1.4 -- Show the difference between end-of-century (2071-2100) and present-day (1981-2010) extreme precipitation across all models

Part 2 -- Internal Variability:
- Part 2.1 -- Get and process the Analytics Engine models; compute statistics
- Part 2.2 -- Visualize annual maximum precipitation through end-of-century across for each ensemble member, with the downscaled data and its corresponding ensemble member highlighted.
- Part 2.3 (coming soon) -- Show the difference between end-of-century (2071-2100) and present-day (1981-2010) extreme precipitation across all ensemble members



# Part 0: Setup

In [ ]:
%%capture
!pip install -r requirements.txt
!pip install xmip
# !pip install seaborn
import climakitae as ck
# import seaborn as sns
import xarray as xr
import pandas as pd
import numpy as np
import datetime
import intake
import dask
import matplotlib as mpl
import matplotlib.pyplot as plt
import xesmf as xe
import cartopy.crs as ccrs
import holoviews as hv
hv.extension('bokeh')

In [ ]:
%%capture
from xmip.preprocessing import (
    rename_cmip6
) # will figure out how to avoid later

def cf_to_dt(ds):
    """
    convert cftime to pandas datetime -
    some GCMs have some wacky calendars
    """
    ds = ds.copy()
    if (
        type(ds.indexes['time']) not in 
        [pd.core.indexes.datetimes.DatetimeIndex]
    ): 
        datetimeindex = ds.indexes['time'].to_datetimeindex()
        ds['time'] = datetimeindex
    return ds

def calendar_align(ds):
    '''
    different models have different calendars
    (some no leap, some 360 day). this results
    in a huge dataset with lots of empty
    values in time when concatenated. 
    the following function sets the day for all monthly
    values to the 1st of each month -
    WARNING this can impact functions which use
    the number of days in each month (eg
    precip flux to total monthly accumulation).
    I can't find a nice way around this.
    '''
    ds['time'] = pd.to_datetime(ds.time.dt.strftime('%Y-%m'))
    return ds

def precip_flux_to_total(ds):
    """
    converts precip flux units 
    (kg m-2 s-1) to total precip
    per month (mm)
    NOTE: assumes regular calendar
    """
    ds = ds.copy()
    ds_attrs = ds.attrs
    days_month = ds.time.dt.days_in_month
    seconds_month = 86400*days_month
    ds = ds*seconds_month
    ds.attrs = ds_attrs
    ds.pr.attrs["units"] = 'mm'
    return ds


def wrapper(ds):
    ds_simulation = ds.attrs["source_id"]
    ds_scenario = ds.attrs["experiment_id"]
    ds_freq = ds.attrs["frequency"]
    
    ds = ds.copy()
    ds = rename_cmip6(ds)
    # ds = broadcast_lonlat(ds)
    # ds = replace_x_y_nominal_lat_lon(ds)
    ds = cf_to_dt(ds)
    if ds_freq in ('mon'):
        ds = calendar_align(ds)
    ds = precip_flux_to_total(ds)
    ds = ds.drop_vars(["lon","lat"],
                     errors="ignore")
    ds = ds.assign_coords({'simulation' : 
                           ds_simulation,
                          'scenario' :
                          ds_scenario})
    ds = ds.squeeze(drop=True)
    
    return ds

In [ ]:
shp_cat = intake.open_catalog(
    "https://cadcat.s3.amazonaws.com/parquet/catalog.yaml")

us_states = shp_cat.states.read()
us_counties = shp_cat.counties.read()
us_watersheds = shp_cat.huc8.read()

shp_cat = None

col = intake.open_esm_datastore("https://cadcat.s3.amazonaws.com/tmp/cmip6-regrid.json")


In [ ]:
class cmip_opt():
    def __init__(self, variable='tas', 
                  area_subset='states', 
                 location='California',
                 timescale='monthly',
                area_average=True):
        self.variable = variable
        self.area_subset = area_subset
        self.location = location
        self.area_average = area_average
        self.timescale = timescale
        
    def cmip_clip(self, ds):
        ds = ds.copy()
        variable = self.variable
        location = self.location
        area_average = self.area_average
        area_subset = self.area_subset
        timescale = self.timescale
        
        to_drop = [v for v in list(
                    ds.data_vars) 
                  if v != variable]
        ds = ds.drop_vars(to_drop)
        ds = clip_region(ds,area_subset,location)
        if area_average:
            ds = area_wgt_average(ds)
        return ds
    
def clip_region(ds,area_subset,location):
    """
    clips CMIP6 dataset using a polygon.
    ds is the dataset
    state is a string ("California")
    (check catalog for other names)
    opt = 'True' to burn in all cells
    which touch the boundary (keep as False)
    """
    ds = ds.copy()
    ds = ds.rio.write_crs(4326)
    
    if 'counties' in area_subset:
        ds_region = us_counties[us_counties.NAME 
                    == location].geometry
    elif 'states' in area_subset:
        ds_region = us_states[us_states.NAME 
                    == location].geometry
        
    try:
        ds = ds.rio.clip(
            geometries=ds_region,crs=4326, drop=True,
        all_touched=False)
    except: 
        #if no grid centers in region
        # instead select all cells which 
        # intersect the region
        print('selecting all cells which intersect region')
        ds = ds.rio.clip(
            geometries=ds_region,crs=4326, drop=True,
        all_touched=True)
        
    return ds

def area_wgt_average(ds):
    ds = ds.copy()
    weights = np.cos(np.deg2rad(ds.y))
    weights.name = "weights"
    ds_weighted = ds.weighted(weights)
    ds = ds_weighted.mean(("x","y"))
    return ds

## set options for CMIP6 data

In [ ]:
copt = cmip_opt()
copt.variable = 'pr'
copt.area_subset = 'counties' 
copt.location = 'Sonoma County'
copt.area_average = True
copt.timescale = 'monthly'

sim_names = {
    "CESM2": "cesm2",
    "CNRM-ESM2-1": "cnrm-esm2-1",
    "EC-Earth3-Veg": "ec-earth3-veg",
    "FGOALS-g3": "fgoals-g3"
}

cmip_names = list(sim_names.keys())

cae_colors_list = [
    '#4477AA', '#66CCEE',
    '#228833', '#EE6677',
    '#AA3377' ,
]

# Part 1: Model Uncertainty

## Part 1.1: Get and process CMIP6 data

In [ ]:
# get the catalogs
cat = col.search(
    table_id = "Amon",
    variable_id = copt.variable,
    experiment_id = ["historical","ssp370"],
    member_id = "r1i1p1f1",
    require_all_on="source_id"
).search(
    activity_id = ["CMIP","ScenarioMIP"]
)

cesm_member = col.search(
    table_id = "Amon",
    variable_id = copt.variable,
    experiment_id = ["historical","ssp370"],
    source_id = "CESM2",
    member_id = "r11i1p1f1",
).search(
    activity_id = ["CMIP","ScenarioMIP"]
)

cnrm_member = col.search(
    table_id = "Amon",
    variable_id = copt.variable,
    experiment_id = ["historical","ssp370"],
    source_id = "CNRM-ESM2-1",
    member_id = "r1i1p1f2",
).search(
    activity_id = ["CMIP","ScenarioMIP"]
)

# get the datasets
cesm_dset = cesm_member.to_dataset_dict(
    zarr_kwargs={'consolidated': True}, 
    storage_options={'anon': True},
    preprocess=wrapper
)

cnrm_dset = cnrm_member.to_dataset_dict(
    zarr_kwargs={'consolidated': True}, 
    storage_options={'anon': True},
    preprocess=wrapper
)
cnrm_dset.update(cesm_dset)

dsets = cat.to_dataset_dict(
    zarr_kwargs={'consolidated': True}, 
    storage_options={'anon': True},
    preprocess=wrapper
)
dsets.update(cnrm_dset)


# Subsets the historical scenario
hist_dsets = {key: val for key,val in dsets.items()
             if "historical" in key}

# Subsets the future scenario
ssp_dsets = {key: val for key,val in dsets.items()
               if "ssp370" in key}

In [ ]:
hist_ds = xr.concat(
    list(hist_dsets.values()),
    dim='simulation').squeeze()
hist_ds = copt.cmip_clip(
    hist_ds).mean(dim='member_id')

# baseline_ds = hist_ds.sel(
#     time=slice('1850','1980'))
hist_ds = hist_ds.sel(
    time=slice('1980','2014'))

ssp_ds = xr.concat(
    list(ssp_dsets.values()),
    dim='simulation').squeeze()
# ssp_ds = ssp_ds.sel(time=slice(
#     '2015','2100'))
ssp_ds = copt.cmip_clip(
    ssp_ds).mean(dim='member_id')

# compute annual maximum and anomaly for each model
# baseline_max = baseline_ds.groupby("time.year").max(dim="time").compute()
# baseline_mean = baseline_max.mean(dim='year').compute()

hist_max = hist_ds.groupby("time.year").max(dim="time").compute()
# hist_max_anom = (hist_max - baseline_mean).compute()

ssp_max = ssp_ds.groupby("time.year").max(dim="time").compute()
# ssp_max_anom = (ssp_max - baseline_mean).compute()

# multimodel means
hist_mmm = hist_max.mean(dim='simulation').compute()
ssp_mmm = ssp_max.mean(dim='simulation').compute()

# hist_mma = hist_max_anom.mean(dim='simulation').compute()
# ssp_mma = ssp_max_anom.mean(dim='simulation').compute()

## Part 1.2: Visualize CMIP6 Model Spread

In [ ]:
hv.extension('bokeh') # need this here for the plots to show 
ind_color = 'grey'
ssp_color = 'orange'
lw = 0.75
alpha = 0.25

all_hist = hist_max.hvplot.line(x='year', by='simulation',legend=False,
                         line_width=lw, color=ind_color, alpha=alpha)
all_ssp = ssp_max.hvplot.line(x='year', by='simulation',legend=False,
                         line_width=lw, color=ssp_color, alpha=alpha)

mmm_hist = hist_mmm.hvplot.line(x='year', label='historical',
                         line_width=lw*2.5, color='black')
mmm_ssp = ssp_mmm.hvplot.line(x='year', label='SSP3-7.0',
                         line_width=lw*2.5, color='orange',
                         title="CMIP6 annual maximum monthly "+
                              "precipitation accumulation")

(all_hist * all_ssp * mmm_hist * mmm_ssp).opts(
    legend_position='left',width=900, height=400)

## 1.3: Highlight the Analytics Engine Models

In [ ]:
hist_cae_models = hist_max.sel(simulation=cmip_names).compute()
ssp_cae_models = ssp_max.sel(simulation=cmip_names).compute()

# get model spread
hist_model_min = hist_max.min(dim='simulation').compute(
).rename({'pr':'minimum'})
hist_model_max = hist_max.max(dim='simulation').compute(
).rename({'pr':'maximum'})

hist_spread = xr.merge([hist_model_min,
                       hist_model_max])

ssp_model_min = ssp_max.min(dim='simulation').compute(
).rename({'pr':'minimum'})
ssp_model_max = ssp_max.max(dim='simulation').compute(
).rename({'pr':'maximum'})

ssp_spread = xr.merge([ssp_model_min,
                       ssp_model_max])

cae_models = xr.concat([hist_cae_models,ssp_cae_models],dim='year')

In [ ]:
hist_spread_plot = hist_spread.hvplot.area(y='minimum', y2='maximum',
                                alpha=alpha, color=ind_color,
                                label = 'historical spread')
ssp_spread_plot = ssp_spread.hvplot.area(y='minimum', y2='maximum',
                                alpha=alpha, color=ssp_color,
                                label='SSP3-7.0 spread')
# ?? sometimes will not use the supplied colors ??
cae_plot = cae_models.hvplot.line(x='year', by='simulation',
                         line_width=lw*2, alpha=0.1, 
                        palette=cae_colors_list)
# sometimes will use supplied colors, just have to plot twice
cae_plot_2 = cae_models.hvplot.line(x='year', by='simulation',
                         line_width=lw*2, alpha=alpha*3, 
                        palette=cae_colors_list)

(hist_spread_plot * ssp_spread_plot * mmm_hist * mmm_ssp * cae_plot * cae_plot_2
).opts(legend_position='left', width=900, height=400)

<span style="color:blue"><i>Is the figure too busy for you?</i> Clicking on a legend entry will dim the corresponding graphic.</span>

## Part 1.4: Change in 99th percentile precipitation accumulation
End-of-century (2071-2100) - present-day (1981-2010)

In [ ]:
hist_percentile = hist_ds.sel(
    time=slice('1981','2010')).chunk(dict(
    time=-1)).quantile([.99],
    dim='time').compute()

ssp_percentile = ssp_ds.sel(
    time=slice('2071','2100')).chunk(dict(
    time=-1)).quantile([.99],
    dim='time').compute()

hist_percentile = hist_percentile.rename(
    dict(quantile='All Models'))
ssp_percentile = ssp_percentile.rename(
    dict(quantile='All Models'))

delta_percentile = (ssp_percentile - hist_percentile).compute()

delta_percentile = delta_percentile.assign_coords(
    {"models":"All CMIP6 models"}).drop(
    "All Models")
     
cmip_percentiles = delta_percentile.sel(
    simulation = cmip_names).assign_coords(
    {"models":"All Cal-Adapt models"}).compute()

indv_percentiles = [cmip_percentiles.sel(simulation=cm
                        ).assign_coords({"models":
                    cm}) for cm in cmip_names] 

indv_percentiles = xr.concat(indv_percentiles,
                             dim="simulation"
                            ).compute()

gcm_percentiles = xr.concat([delta_percentile,
          cmip_percentiles,indv_percentiles],
         dim = 'simulation').compute()

In [ ]:
cmip_delta = gcm_percentiles.hvplot.scatter(
    y='pr',x="models",by='simulation',
    legend=False,title=("Change in 99th percentile"
                        +" monthly precipitation accumulation")
)

cmip_delta  

# Part 2: Internal Variability

## 2.1: Get and process Analytics Engine models

### 2.1.1: First the CMIP6 models

In [ ]:
cat = col.search(
    table_id = "Amon",
    variable_id = 'pr',
    experiment_id = ["historical","ssp370"],
    source_id = cmip_names,
)
dsets = cat.to_dataset_dict(zarr_kwargs={'consolidated': True}, 
                            storage_options={'anon': True},
                           preprocess=wrapper
                           )

# Subsets the historical scenario
hist_dsets = {key: val for key,val in dsets.items()
             if "historical" in key}

# Subsets the future scenario
ssp_dsets = {key: val for key,val in dsets.items()
               if "ssp370" in key}

cae_ds = xr.concat(
    list(dsets.values()),
    dim='simulation').squeeze()
cae_ds = copt.cmip_clip(
    cae_ds)

### 2.1.2: Then get their downscaled counterparts

In [ ]:
app = ck.Application()
app.selections.scenario_historical=['Historical Climate']
app.selections.scenario_ssp=['SSP 3-7.0 -- Business as Usual']
app.selections.append_historical = True
app.selections.area_average = copt.area_average
app.selections.variable = 'Precipitation (total)'
app.selections.time_slice = (1981, 2100)
app.selections.resolution = '45 km'
app.location.area_subset='CA counties'
app.location.cached_area = copt.location

wrf_ds = app.retrieve()

## 2.2: Show the spread across ensemble members, including the downscaled data

In [ ]:
cmip_wrf_members = {
    "CESM2": "r11i1p1f1",
    "CNRM-ESM2-1": "r1i1p1f2",
    "EC-Earth3-Veg": "r1i1p1f1",
    "FGOALS-g3": "r1i1p1f1",
    # "MPI-ESM1-2-LR": "r7i1p1f1",
}
wrf_max = wrf_ds.groupby("time.year").max(dim="time").squeeze().compute()
cae_max = cae_ds.groupby("time.year").max(dim="time").compute()
cae_wrf_member = cae_max.sel(
    member_id = list(cmip_wrf_members.values()))

renamed_ds_list = []

for cn,wn in sim_names.items():
    renamed_ds_list.append(
        wrf_max.sel(simulation=wn
                  ).squeeze().assign_coords(
            {'simulation' : cn}))
    
wrf_max = xr.concat(renamed_ds_list,dim='simulation')

In [ ]:
# hv.extension('bokeh')
ind_color = 'grey'
ssp_color = 'orange'
lw = 0.75
alpha = 0.25

# all_cae_max = cae_max.sel(year=slice(1981,2100)).hvplot.line(x='year', by='member_id',
#                          line_width=lw, color=ind_color, alpha=alpha*3,
#                         legend=False)

hist_cae_max = cae_max.sel(year=slice(1981,2014)).hvplot.line(x='year', by='member_id',
                         line_width=lw, color=ind_color, alpha=alpha*3,
                        legend=False)
ssp_cae_max = cae_max.sel(year=slice(2015,2100)).hvplot.line(x='year', by='member_id',
                         line_width=lw, color=ssp_color, alpha=alpha*3,
                        legend=False)

wrf_member = wrf_max.squeeze().hvplot.line(x='year',  
                        line_width=lw*2, color='blue', alpha=alpha*3,
                        label = "Downscaled member")

cae_member = cae_wrf_member.sel(year=slice(1981,2100)).mean(
                        dim='member_id').hvplot.line(x='year',  
                        line_width=lw*2, color='magenta', alpha=alpha*3,
                        label = "Corresponding CMIP6 ensemble member")


(hist_cae_max *  ssp_cae_max * wrf_member * cae_member).opts(
    legend_position='top',width=900, height=400) 

<span style="color:red">Notes for our in-house reviewers:</span> 

1. Yes that plot is buggy - will fix later
2. There will be one more plot after this (see the storyboard). I am having trouble getting the dataset to cooperate, and I am tired, and will get to it on Monday.
3. Also will include actual guidance based off the results.

### IGNORE code to (maybe) use later

In [ ]:

# ?? this does not use the supplied palette at all??
# hist_cae_1 = hist_cae_models.hvplot.line(x='year', by='simulation',
#                          line_width=lw*2, alpha=0.1, 
#                         palette=cae_colors_list)
# hist_cae_2 = hist_cae_models.hvplot.line(x='year', by='simulation',
#                          line_width=lw*2, alpha=0.1, 
#                         palette=cae_colors_list)
# hist_cae = hist_cae_1 * hist_cae_2
# ssp_cae_1 = ssp_cae_models.hvplot.line(x='year', by='simulation',
#                          line_width=lw*2, alpha=alpha*2, 
#                         palette=cae_colors_list)
# ssp_cae_2 = ssp_cae_models.hvplot.line(x='year', by='simulation',
#                          line_width=lw*2, alpha=alpha*2, 
#                         palette=cae_colors_list)
# ssp_cae = ssp_cae_1 * ssp_cae_2

In [ ]:
# # percentile for the cal-adapt GCMs (together)
# pd_cae_ds = cae_ds.sel(
#     time=slice('1981','2010'))
# eoc_cae_ds = cae_ds.sel(time=slice(
#     '2071','2100'))
# pd_cae_percentile = pd_cae_ds.chunk(dict(
#     time=-1)).quantile([.99],
#     dim='time').compute()
# eoc_cae_percentile = eoc_cae_ds.chunk(dict(
#     time=-1)).quantile([.99],
#     dim='time').compute()
# pd_cae_percentile = pd_cae_percentile.rename(
#     dict(quantile='All Models'))
# eoc_cae_percentile = eoc_cae_percentile.rename(
#     dict(quantile='All Models'))
# cae_delta_percentile = (eoc_cae_percentile 
#                     - pd_cae_percentile).compute()
# cae_delta_percentile = delta_percentile.assign_coords(
#     {"models":"CMIP6 ensemble members"}).drop(
#     "All Models")

# # percentile for the cal-adapt GCMs (individually)
# cae_indv_percentiles = [cae_delta_percentile.sel(
#                     simulation=cm).assign_coords({"models":
#                     cm}) for cm in cmip_names] 
# cae_indv_percentiles = xr.concat(cae_indv_percentiles,
#                         dim="simulation").compute()
# cae_percentiles = xr.concat([cae_delta_percentile,
#                     cae_indv_percentiles],
#                     dim = 'simulation').compute()

In [ ]:
cae_delta_percentile

In [ ]:
hist_wrf = wrf_ds.sel(time=slice('1980','2014'))
ssp_wrf = wrf_ds.sel(time=slice('2015','2100'))

In [ ]:
# compute annual maximum for each model
# first the downscaled
# hist_wrf_max = hist_wrf.groupby("time.year").max(dim="time").compute()
# ssp_wrf_max = ssp_wrf.groupby("time.year").max(dim="time").compute()
# then the original
# hist_cae_max = hist_cae_ds.groupby("time.year").max(dim="time").compute()
# ssp_cae_max = ssp_cae_ds.groupby("time.year").max(dim="time").compute()
# now extract the ensemble members
# which have downscaled counterpart
# hist_cae_wrf_member = hist_cae_max.sel(
#     member_id = list(cmip_wrf_members.values()))

# ssp_cae_wrf_member = ssp_cae_max.sel(
#     member_id = list(cmip_wrf_members.values()))

In [ ]:
# # hv.extension('bokeh')
# ind_color = 'grey'
# ssp_color = 'orange'
# lw = 0.75
# alpha = 0.25

# all_hist = hist_cae_max.hvplot.line(x='year', by='member_id',label="CMIP6 ensemble members",
#                          line_width=lw, color=ind_color, alpha=alpha*3,
#                         legend=False)
# all_ssp = ssp_cae_max.hvplot.line(x='year', by='member_id',
#                          line_width=lw, color=ind_color, alpha=alpha*3,
#                             legend=False)

# hist_cae_member = hist_cae_wrf_member.hvplot.line(x='year', by='member_id', 
#                         line_width=lw*2, color='purple', alpha=alpha*3,
#                         label = "CMIP6 member which was downscaled")

# ssp_cae_member = ssp_cae_wrf_member.hvplot.line(x='year', by='member_id', 
#                          line_width=lw*2, color='purple', alpha=alpha*3,
#                         label = "Downscaled results")

# # mmm_hist = hist_mmm.hvplot.line(x='year', label='historical',
# #                          line_width=lw*2.5, color='black')
# # mmm_ssp = ssp_mmm.hvplot.line(x='year', label='SSP3-7.0',
# #                          line_width=lw*2.5, color='orange',
# #                          title="CMIP6 annual maximum monthly "+
# #                               "precipitation accumulation")

# (all_hist * all_ssp * hist_cae_member * ssp_cae_member).opts(
#     legend_position='left',width=900, height=400)

In [ ]:
# hv.extension('bokeh')
# ind_color = 'grey'
# ssp_color = 'orange'
# lw = 0.75
# alpha = 0.5

# all_hist_anom = hist_max_anom.hvplot.line(x='year', by='simulation',
#                          line_width=lw, color=ind_color, alpha=alpha, legend=False)
# all_ssp_anom = ssp_max_anom.hvplot.line(x='year', by='simulation',
#                          line_width=lw, color=ssp_color, alpha=alpha, legend=False)
# mma_hist = hist_mma.hvplot.line(x="year",
#                          line_width=lw*2.5, color='black', legend=False)
# mma_ssp = ssp_mma.hvplot.line(x="year",
#                          line_width=lw*2.5, color='red', legend=False,
#                          title="CMIP6 annual maximum precipitation "+
#                               "accumulation relative to 1850-1900 maximum")

# anom_plot = all_hist_anom * all_ssp_anom * mma_hist * mma_ssp

# anom_plot